# Analyze SH17 PP-PicoDet-L Variants

This notebook reads real PP-PicoDet-L training/evaluation artifacts and prepares report-friendly CSV/PNG outputs. Missing metrics stay as `pending`; the notebook does not synthesize or fake scores.


In [ ]:
from pathlib import Path
import csv
import os
import re

import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd()
OUTPUT_ROOT = Path(os.environ.get("SH17_OUTPUT_ROOT", PROJECT_ROOT / "outputs" / "picodet_l_paddledet")).expanduser()
SUMMARY_METRICS_CSV = OUTPUT_ROOT / "summary_metrics.csv"
PER_CLASS_METRICS_CSV = OUTPUT_ROOT / "per_class_metrics.csv"
LOG_DIR = OUTPUT_ROOT / "logs"
ANALYSIS_DIR = OUTPUT_ROOT / "analysis"
FIGURE_DIR = ANALYSIS_DIR / "figures"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

VARIANT_ORDER = [
    "picodet_l_320_baseline_50e",
    "picodet_l_640_scale_50e",
    "picodet_l_640_oversample_minority_50e",
    "picodet_l_640_zoom_crop_50e",
]

VARIANT_LABELS = {
    "picodet_l_320_baseline_50e": "Baseline 320",
    "picodet_l_640_scale_50e": "Scale 640",
    "picodet_l_640_oversample_minority_50e": "Oversample 640",
    "picodet_l_640_zoom_crop_50e": "Zoom-crop 640",
}

MINORITY_CLASSES = ["ear-mufs", "face-guard", "foot", "helmet", "medical-suit", "safety-vest"]

print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("SUMMARY_METRICS_CSV:", SUMMARY_METRICS_CSV)
print("PER_CLASS_METRICS_CSV:", PER_CLASS_METRICS_CSV)


In [ ]:
def read_csv_rows(path):
    if not path.exists():
        return []
    with path.open("r", newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


def pending_summary_rows():
    rows = []
    for name in VARIANT_ORDER:
        input_size = 320 if "320" in name else 640
        batch_size = 24 if input_size == 320 else 12
        base_lr = 0.12 if input_size == 320 else 0.06
        train_json = "instances_train_oversampled.json" if "oversample" in name else "instances_train.json"
        rows.append({
            "experiment": name,
            "input_size": str(input_size),
            "params_m": "5.80",
            "batch_size": str(batch_size),
            "base_lr": str(base_lr),
            "epochs": "50",
            "train_json": train_json,
            "map50_95": "pending",
            "map50": "pending",
            "precision": "pending",
            "recall": "pending",
            "status": "pending",
            "bbox_json": str(OUTPUT_ROOT / "eval" / name / "bbox.json"),
        })
    return rows


def to_float_or_none(value):
    try:
        if value is None or str(value).strip().lower() == "pending":
            return None
        return float(value)
    except ValueError:
        return None


summary_rows = read_csv_rows(SUMMARY_METRICS_CSV) or pending_summary_rows()
summary_by_name = {row["experiment"]: row for row in summary_rows}
ordered_summary_rows = [summary_by_name.get(name) for name in VARIANT_ORDER if name in summary_by_name]

print("Loaded summary rows:", len(ordered_summary_rows))
for row in ordered_summary_rows:
    print(row["experiment"], row.get("status", "pending"), row.get("map50_95", "pending"), row.get("map50", "pending"))


In [ ]:
def save_table_image(rows, columns, output_path, title):
    display_rows = [[row.get(column, "pending") for column in columns] for row in rows]
    fig_height = max(2.4, 0.42 * (len(display_rows) + 2))
    fig, ax = plt.subplots(figsize=(12, fig_height))
    ax.axis("off")
    ax.set_title(title, fontsize=14, pad=14)
    table = ax.table(
        cellText=display_rows,
        colLabels=columns,
        loc="center",
        cellLoc="center",
    )
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.25)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


best_table_rows = []
for row in ordered_summary_rows:
    best_table_rows.append({
        "Variant": VARIANT_LABELS.get(row["experiment"], row["experiment"]),
        "Input": row.get("input_size", "pending"),
        "Params(M)": row.get("params_m", "5.80"),
        "mAP50-95": row.get("map50_95", "pending"),
        "mAP50": row.get("map50", "pending"),
        "Recall": row.get("recall", "pending"),
        "Status": row.get("status", "pending"),
    })

variant_best_metrics_table = FIGURE_DIR / "variant_best_metrics_table.png"
save_table_image(
    best_table_rows,
    ["Variant", "Input", "Params(M)", "mAP50-95", "mAP50", "Recall", "Status"],
    variant_best_metrics_table,
    "PP-PicoDet-L Variant Metrics",
)
print("Saved:", variant_best_metrics_table)


In [ ]:
def save_metric_bar_chart(rows, metric_key, output_path, title, ylabel):
    labels = [VARIANT_LABELS.get(row["experiment"], row["experiment"]) for row in rows]
    values = [to_float_or_none(row.get(metric_key)) for row in rows]
    plot_values = [0 if value is None else value for value in values]
    colors = ["#9aa0a6" if value is None else "#2f80ed" for value in values]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(labels, plot_values, color=colors)
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_ylim(0, max([1.0, *plot_values]))
    ax.tick_params(axis="x", rotation=15)
    for bar, value in zip(bars, values):
        text = "pending" if value is None else f"{value:.3f}"
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02, text, ha="center", va="bottom", fontsize=9)
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)


variant_map_comparison = FIGURE_DIR / "variant_map_comparison.png"
save_metric_bar_chart(
    ordered_summary_rows,
    "map50_95",
    variant_map_comparison,
    "PP-PicoDet-L mAP50-95 Comparison",
    "mAP50-95",
)
print("Saved:", variant_map_comparison)


In [ ]:
def train_loss_points(experiment_name):
    log_path = LOG_DIR / f"train_{experiment_name}.log"
    if not log_path.exists():
        return []
    pattern = re.compile(r"Epoch:\s*\[(\d+)\].*?loss:\s*([0-9.]+)")
    points = []
    for line in log_path.read_text(encoding="utf-8", errors="replace").splitlines():
        match = pattern.search(line)
        if match:
            points.append((int(match.group(1)), float(match.group(2))))
    return points


fig, ax = plt.subplots(figsize=(10, 5))
has_points = False
for experiment_name in VARIANT_ORDER:
    points = train_loss_points(experiment_name)
    if not points:
        continue
    has_points = True
    epochs = [epoch for epoch, _ in points]
    losses = [loss for _, loss in points]
    ax.plot(epochs, losses, label=VARIANT_LABELS.get(experiment_name, experiment_name))

if has_points:
    ax.set_title("PP-PicoDet-L Training Loss")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.legend()
else:
    ax.axis("off")
    ax.text(0.5, 0.5, "pending: train logs not found", ha="center", va="center", fontsize=14)

training_loss_curves = FIGURE_DIR / "training_loss_curves.png"
fig.tight_layout()
fig.savefig(training_loss_curves, dpi=180, bbox_inches="tight")
plt.close(fig)
print("Saved:", training_loss_curves)


In [ ]:
per_class_rows = read_csv_rows(PER_CLASS_METRICS_CSV)
if not per_class_rows:
    per_class_rows = [
        {
            "experiment": experiment,
            "class_name": class_name,
            "ap": "pending",
            "precision": "pending",
            "recall": "pending",
            "status": "pending",
        }
        for experiment in VARIANT_ORDER
        for class_name in MINORITY_CLASSES
    ]

minority_rows = []
for row in per_class_rows:
    if row.get("class_name") in MINORITY_CLASSES and row.get("experiment") in VARIANT_ORDER:
        minority_rows.append({
            "Variant": VARIANT_LABELS.get(row["experiment"], row["experiment"]),
            "Class": row.get("class_name", "pending"),
            "AP": row.get("ap", "pending"),
            "Precision": row.get("precision", "pending"),
            "Recall": row.get("recall", "pending"),
            "Status": row.get("status", "pending"),
        })

minority_class_metrics_table = FIGURE_DIR / "minority_class_metrics_table.png"
save_table_image(
    minority_rows,
    ["Variant", "Class", "AP", "Precision", "Recall", "Status"],
    minority_class_metrics_table,
    "Minority-Class PP-PicoDet-L Metrics",
)
print("Saved:", minority_class_metrics_table)
